
# HerWILL Safe Social Media Datathon 2026 — Harmful Content Classification

**Task:** 3-class text classification of Bangla / English / Banglish (code-mixed) social
media posts into:

| Label | Meaning |
|---|---|
| `0` | Explicitly Toxic / Hateful |
| `1` | Subtly Toxic |
| `2` | Non-toxic / Neutral |

**Metric:** Macro F1.

**Approach:** Fine-tune a multilingual transformer (`XLM-RoBERTa-base`) which natively
handles Bangla, English, and code-mixed "Banglish" text without needing translation
(see the Data Cleaning section for why we deliberately do **not** translate).
We train with **Stratified K-Fold cross-validation**, **class-weighted loss** to
counter class imbalance, and **ensemble the fold models** (average softmax
probabilities) for the final, more robust test prediction. We also apply a safe
post-processing step that reuses the *known* training label whenever a test post's
text is an exact duplicate of a training post.

> **Before running on Kaggle:**
> 1. Add your competition dataset to this notebook (Add Data → your dataset).
> 2. Edit `DATA_DIR` in the Config cell below to point at the folder containing
>    `train.csv` and `test.csv`.
> 3. Turn ON **GPU** (Settings → Accelerator → GPU T4 x2 or P100) and **Internet**
>    (Settings → Internet → On) so the pretrained XLM-R weights can be downloaded.


## 1. Setup

In [ ]:

# If a package is missing on the Kaggle image, this installs it quietly.
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

ensure("transformers")
ensure("datasets")
ensure("scikit-learn", "sklearn")


In [ ]:

import os, re, gc, random, unicodedata, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU detected. Turn on GPU in Notebook Settings for reasonable training time.")


## 2. Config — EDIT `DATA_DIR` FOR YOUR KAGGLE DATASET

In [ ]:

# --------------------------------------------------------------------------
# EDIT THIS to match the path of your dataset on Kaggle, e.g.
#   "/kaggle/input/herwill-safe-social-media-datathon-2026"
# You can see the exact path in the "Data" pane on the right side of the
# Kaggle notebook editor after you add the competition dataset.
# --------------------------------------------------------------------------
DATA_DIR = "/kaggle/input/herwill-safe-social-media-datathon-2026"

TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH  = os.path.join(DATA_DIR, "test.csv")

OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Model / training config -------------------------------------------------
MODEL_NAME = "xlm-roberta-base"   # multilingual: handles Bangla + English + code-mix
MAX_LEN = 128                     # covers ~99% of posts after cleaning (see EDA)
N_FOLDS = 5                       # set to e.g. 3 if you need a faster run
EPOCHS = 3
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
NUM_LABELS = 3
SEED = 42

# Quick sanity check that the files exist before we go further.
assert os.path.exists(TRAIN_PATH), f"train.csv not found at {TRAIN_PATH} -- fix DATA_DIR above."
assert os.path.exists(TEST_PATH), f"test.csv not found at {TEST_PATH} -- fix DATA_DIR above."
print("Found train.csv and test.csv OK.")


## 3. Load Data

In [ ]:

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("train shape:", train_df.shape)
print("test shape :", test_df.shape)
train_df.head()


## 4. Exploratory Data Analysis — what's actually wrong with this data?

Before modeling, we inspect the raw files closely (as requested) rather than
assuming they're clean. Running this on the full `train.csv` surfaces several
real problems that would otherwise quietly hurt the model:


In [ ]:

print("Missing values (train):\n", train_df.isnull().sum(), sep="")
print("\nMissing values (test):\n", test_df.isnull().sum(), sep="")

print("\nClass distribution (train):")
print(train_df["y"].value_counts())
print((train_df["y"].value_counts(normalize=True) * 100).round(2).astype(str) + "%")

print("\nExact duplicate (text, y) rows:", train_df.duplicated(subset=["text", "y"]).sum())
print("Duplicate ids:", train_df["id"].duplicated().sum())

# Texts that appear more than once with DIFFERENT labels -> label noise
dup_texts = train_df[train_df.duplicated("text", keep=False)]
label_conflicts = dup_texts.groupby("text")["y"].nunique()
conflicting_texts = label_conflicts[label_conflicts > 1]
print("Texts with conflicting labels across duplicate rows:", len(conflicting_texts))

# Extremely long outlier rows: scraped multi-comment threads mislabeled as a
# single post, not representative social-media posts.
lens = train_df["text"].astype(str).str.len()
print("\nText length describe (train):")
print(lens.describe())
print("Rows with text length > 2000 chars:", (lens > 2000).sum())

# Train/test text leakage: identical posts appear in both splits.
overlap = set(train_df["text"]) & set(test_df["text"])
print("\nIdentical text present in BOTH train and test:", len(overlap))



### Problems found, and how we fix each one

| # | Problem | Evidence | Fix |
|---|---|---|---|
| 1 | **Class imbalance** | class 0 ≈ 23.8k, class 1 ≈ 18.2k, class 2 ≈ 5.9k (~4x rarer) | class-weighted loss during fine-tuning (computed from the *cleaned* label distribution) |
| 2 | **Exact duplicate rows** | ~568 rows share identical `(text, y)` | drop duplicates, keep first occurrence |
| 3 | **Conflicting-label duplicates** | ~52 texts (e.g. `"Fake news"`, `"Disgusting"`) appear more than once with *different* labels — label noise from annotation | resolve by **majority vote**; if perfectly tied, drop the row (ambiguous, would only inject noise) |
| 4 | **Extreme outlier rows** | a handful of rows are >10,000 characters — entire scraped comment-section / chat-export threads (some even in Arabic), not single social posts, mislabeled as class 0 | drop these from **training only** (length > 2000 chars); test rows are always kept (every test id must get a prediction) and simply truncated by the tokenizer |
| 5 | **Whitespace / unicode noise** | inconsistent spacing, unnormalized unicode | normalize with NFC + collapse whitespace |
| 6 | **Train/test text leakage** | ~254 identical texts appear in both train and test | after modeling, directly reuse the (cleaned, majority-voted) training label for any test row whose text exactly matches a training row — this is legitimate here since it's the same underlying label, not overfitting |
| 7 | **Code-mixed Bangla/English ("Banglish")** | ~990 rows mix Bangla and Latin script in the same post, plus Bangla-only and English-only rows | use a **multilingual** model (XLM-R) instead of a Bangla-only or English-only model, and deliberately **do not machine-translate** — see note below |

**Why we don't translate everything into Bangla:** the prompt is `harmful content` in
`code-mixed, sarcastic, coded language`. Machine translation tends to flatten sarcasm,
slang, and code-switching cues (exactly the signal that separates "Subtly Toxic" from
"Non-toxic"), and would introduce a second source of noise on top of the labels.
XLM-RoBERTa is pretrained on 100 languages including Bengali and English and handles
code-mixed input directly, so we keep the original text as-is (only whitespace/unicode
normalized) and let the multilingual model do the work.


## 5. Clean the Data

In [ ]:

def normalize_text(s: str) -> str:
    s = str(s)
    s = unicodedata.normalize("NFC", s)
    s = s.replace("\u200c", "").replace("\u200d", "")  # stray zero-width joiners
    s = re.sub(r"\s+", " ", s).strip()
    return s

def clean_train(df: pd.DataFrame, max_chars: int = 2000) -> pd.DataFrame:
    df = df.copy()
    df["text"] = df["text"].apply(normalize_text)

    before = len(df)
    df = df.drop_duplicates(subset=["text", "y"], keep="first")
    print(f"Dropped {before - len(df)} exact duplicate (text,y) rows.")

    # Resolve conflicting-label duplicate texts by majority vote (pandas-version
    # safe: computed as a text -> label lookup rather than groupby().apply(),
    # since groupby(...).apply() behavior around the grouping column has changed
    # across pandas versions).
    def majority_label(s):
        vc = s.value_counts()
        if len(vc) == 1 or vc.iloc[0] > vc.iloc[1]:
            return vc.idxmax()
        return np.nan  # true tie -> ambiguous, drop

    label_map = df.groupby("text")["y"].apply(majority_label)

    before = len(df)
    df = df.drop_duplicates(subset=["text"], keep="first").copy()
    df["y"] = df["text"].map(label_map)
    df = df[df["y"].notna()].reset_index(drop=True)
    df["y"] = df["y"].astype(int)
    print(f"Resolved/dropped {before - len(df)} rows from duplicate texts (majority vote or tie-drop).")

    before = len(df)
    df = df[df["text"].str.len() <= max_chars].reset_index(drop=True)
    print(f"Dropped {before - len(df)} extreme-outlier rows (> {max_chars} chars).")

    df = df[df["text"].str.len() > 0].reset_index(drop=True)
    return df

def clean_test(df: pd.DataFrame) -> pd.DataFrame:
    # Never drop test rows -- every id needs a prediction. Only normalize text.
    df = df.copy()
    df["text"] = df["text"].apply(normalize_text)
    df["text"] = df["text"].replace("", "[EMPTY]")
    return df

train_clean = clean_train(train_df)
test_clean = clean_test(test_df)

print("\nFinal train shape:", train_clean.shape)
print("Final class distribution:")
print(train_clean["y"].value_counts())


## 6. Dataset / Tokenizer

In [ ]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ToxicityDataset(Dataset):
    def __init__(self, texts, labels=None, max_len=MAX_LEN):
        self.texts = list(texts)
        self.labels = list(labels) if labels is not None else None
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding=False,
        )
        item = {k: torch.tensor(v) for k, v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


## 7. Metric (Macro F1) + Class-Weighted Trainer

In [ ]:

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    macro_f1 = f1_score(labels, preds, average="macro")
    return {"macro_f1": macro_f1}

class WeightedTrainer(Trainer):
    '''Trainer with a class-weighted CrossEntropy loss to counter the
    imbalance between Explicit (0), Subtle (1) and Neutral (2).'''
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fct = nn.CrossEntropyLoss(weight=weight)
        loss = loss_fct(logits.view(-1, NUM_LABELS), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def make_training_args(fold_out_dir):
    kwargs = dict(
        output_dir=fold_out_dir,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        logging_steps=100,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=SEED,
    )
    # transformers renamed evaluation_strategy -> eval_strategy in newer versions;
    # try the new name first, fall back to the old one for older Kaggle images.
    try:
        return TrainingArguments(evaluation_strategy="epoch", save_strategy="epoch", **kwargs)
    except TypeError:
        return TrainingArguments(eval_strategy="epoch", save_strategy="epoch", **kwargs)


## 8. Stratified K-Fold Training + Test Prediction (ensembled)

In [ ]:

texts = train_clean["text"].values
labels = train_clean["y"].astype(int).values

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

test_dataset = ToxicityDataset(test_clean["text"].values, labels=None)

oof_preds = np.zeros(len(labels), dtype=int)
test_probs_sum = np.zeros((len(test_clean), NUM_LABELS), dtype=np.float64)
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(texts, labels)):
    print(f"\n{'='*60}\nFOLD {fold + 1}/{N_FOLDS}\n{'='*60}")

    X_train, y_train = texts[train_idx], labels[train_idx]
    X_val, y_val = texts[val_idx], labels[val_idx]

    train_ds = ToxicityDataset(X_train, y_train)
    val_ds = ToxicityDataset(X_val, y_val)

    class_weights = compute_class_weight(
        class_weight="balanced", classes=np.arange(NUM_LABELS), y=y_train
    )
    class_weights = torch.tensor(class_weights, dtype=torch.float)
    print("Class weights:", class_weights.tolist())

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS
    )

    fold_dir = os.path.join(OUTPUT_DIR, f"fold_{fold}")
    args = make_training_args(fold_dir)

    trainer = WeightedTrainer(
        class_weights=class_weights,
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=data_collator,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    val_pred = trainer.predict(val_ds)
    val_logits = val_pred.predictions
    val_argmax = np.argmax(val_logits, axis=1)
    oof_preds[val_idx] = val_argmax
    fold_f1 = f1_score(y_val, val_argmax, average="macro")
    fold_scores.append(fold_f1)
    print(f"Fold {fold + 1} macro F1: {fold_f1:.4f}")

    test_pred = trainer.predict(test_dataset)
    test_logits = test_pred.predictions
    test_probs = torch.softmax(torch.tensor(test_logits), dim=1).numpy()
    test_probs_sum += test_probs

    # free memory before the next fold
    del model, trainer
    gc.collect()
    torch.cuda.empty_cache()

print("\n" + "=" * 60)
print("Per-fold macro F1:", [round(s, 4) for s in fold_scores])
print("Mean fold macro F1:", round(np.mean(fold_scores), 4))
print("Overall OOF macro F1:", round(f1_score(labels, oof_preds, average="macro"), 4))
print(classification_report(labels, oof_preds, target_names=["Explicit(0)", "Subtle(1)", "Neutral(2)"]))


## 9. Ensemble Test Predictions + Safe Leakage Post-Processing

In [ ]:

# Average the softmax probabilities across all folds, then take argmax.
test_probs_avg = test_probs_sum / N_FOLDS
final_preds = np.argmax(test_probs_avg, axis=1)

submission = pd.DataFrame({
    "id": test_clean["id"].values,
    "y_pred": final_preds,
})

# --- Reuse known training labels for exact text matches (train/test overlap) ---
# We found ~254 identical posts shared between train and test. For those rows we
# already know the true label from the (cleaned, majority-voted) training set, so
# we trust it over the model's prediction.
text_to_label = dict(zip(train_clean["text"], train_clean["y"].astype(int)))
test_text_lookup = test_clean.set_index("id")["text"].to_dict()

overridden = 0
for i, row_id in enumerate(submission["id"]):
    t = test_text_lookup[row_id]
    if t in text_to_label:
        new_label = text_to_label[t]
        if submission.loc[i, "y_pred"] != new_label:
            overridden += 1
        submission.loc[i, "y_pred"] = new_label

print(f"Overrode {overridden} predictions using exact train/test text matches.")


## 10. Build & Save the Submission File

In [ ]:

# Sanity checks before saving.
assert submission["id"].isin(test_df["id"]).all(), "Some ids don't match test.csv!"
assert set(submission["id"]) == set(test_df["id"]), "Missing/extra ids vs test.csv!"
assert submission["y_pred"].isin([0, 1, 2]).all(), "Found a label outside {0,1,2}!"
assert submission.shape[0] == test_df.shape[0], "Row count mismatch with test.csv!"

# Match the id order of the original test.csv exactly.
submission = submission.set_index("id").loc[test_df["id"]].reset_index()

submission_path = os.path.join(OUTPUT_DIR, "submission.csv")
submission.to_csv(submission_path, index=False)

print("Saved:", submission_path)
print(submission.head())
print("\nPredicted label distribution on test set:")
print(submission["y_pred"].value_counts())



## Notes on further improving the score

- **More folds / epochs**: `N_FOLDS=5, EPOCHS=3` is a strong, time-reasonable default.
  If you have GPU budget left, bumping `EPOCHS` to 4–5 or trying
  `xlm-roberta-large` (bigger, slower, usually a few points better macro F1) can help.
- **Bangla-specialist model ensembling**: you can add a second model such as
  `csebuetnlp/banglabert` (Bangla-only) trained the same way and average its test
  probabilities with the XLM-R ones — code-mixed rows will still be handled by XLM-R,
  while pure-Bangla rows may get an extra boost from a Bangla-specialist model.
- **Threshold/ class-2 recall**: class `2` (Neutral) is the rarest (~12% of data).
  If macro F1 on the OOF report above shows lower recall for class 2, try increasing
  its class weight slightly beyond `balanced`, e.g. multiply the class-2 weight by 1.2–1.5.
- **Label-noise review**: the "conflicting-label" texts and outlier long rows were
  handled automatically above; if you have time, manually reviewing a sample of them
  (printed in Section 4/5) can catch further annotation issues worth relabeling.
